In [1]:
pwd

'/Users/ud4/repos/GitHub/FATESFACE/Jupyter_Notebooks/Paper1_Plots'

In [11]:
df2_stack

,TREE.ID,Species,Plot,Quad,CO2,N,DBH
0,11003,PITA,1,1,amb,cont,18.9
0,11003,PITA,1,1,amb,cont,19.5
0,11003,PITA,1,1,amb,cont,20.4
0,11003,PITA,1,1,amb,cont,21.2
0,11003,PITA,1,1,amb,cont,21.8
...,...,...,...,...,...,...,...
2190,84195,ULAL,8,4,amb,fert,2.4
2190,84195,ULAL,8,4,amb,fert,2.5
2190,84195,ULAL,8,4,amb,fert,2.4
2190,84195,ULAL,8,4,amb,fert,2.5


In [15]:
import pandas as pd
import numpy as np
from datetime import datetime

# Function definitions
def increment(v):
    return v[-1] - v[0]

def increment_mean(v):
    return np.mean(v[-2:]) - np.mean(v[:2])

def final(v):
    return v[-1]

def first(v):
    return v[0]

# Directories and filenames
date = '200209'
wd = '/Volumes/disk2/Research_Projects/FACE_modelling/Phase_4/'
wd = '/Users/ud4/Downloads'
wd_duke = f'{wd}/'
wd_ornl = f'{wd}/'
wdo = f'{wd}/results/{date}'

dk_dbh_fname = 'DukeFACE_allometry_gfD.csv'
or_dbh_fname = 'wood_ts.csv'
df2_dk = df2_stack
print(df2_dk.head())



   RING  TREE  BA
0   1.0   2.0 NaN
1   1.0   3.0 NaN
2   1.0   4.0 NaN
3   1.0   5.0 NaN
4   1.0   6.0 NaN


In [22]:
3.14*(25/2)**2

490.625

### Notes
1. I do not have to factor in the mortality. Since we are comparing it with model output were mortality is also dynamic
2. Make this data to the same shape as Model Simulation
3. Area or ORNL Plots is 314 m2 (pi * ((25m -5m buffer)/2)^2)
4. Area of DUKE Plots is 490.625 m2 (pi * ((30m -5m buffer)/2)^2)

In [19]:
# Read data: ORNL
df2 = pd.read_csv(f'{wd_ornl}{or_dbh_fname}')
df2_stack = pd.DataFrame({'RING': df2['RING'], 'TREE': df2['TREE'], 'BA': df2.iloc[:, 3:94].stack()})
#df2_stack['ptime'] = pd.to_datetime(df2_stack['ind'].str.replace('X', ''), format="%Y.%m.%d")
#df2_stack['co2'] = np.where(df2_stack['RING'].isin([1, 2]), 'ele', 'amb')
#df2_stack['treeid'] = df2_stack['RING'].astype(str) + '.' + df2_stack['TREE'].astype(str)

In [21]:
df2

,RING,TREE,1997-03-11,1997-05-15,1997-06-13,1997-07-21,1997-08-27,1997-10-03,1997-12-10,1998-04-17,...,2008-07-21,2008-08-19,2008-09-25,2008-11-20,2009-04-20,2009-05-12,2009-06-11,2009-07-09,2009-08-18,2009-09-28
0,1,2,52,52,52,52,52,52,52,52,...,0,0,0,0,0,0,0,0,0,0
1,1,3,157,157,161,168,174,175,175,176,...,364,366,367,367,367,368,369,372,373,374
2,1,4,103,103,103,104,105,105,105,106,...,129,129,129,129,129,129,129,129,129,129
3,1,5,125,125,127,131,135,136,136,136,...,246,247,248,248,248,248,249,252,252,252
4,1,6,70,70,70,71,72,72,72,73,...,84,83,84,84,84,84,83,84,83,84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438,5,96,138,139,142,147,154,155,155,156,...,251,251,251,251,251,251,253,254,255,255
439,5,98,83,83,83,84,87,88,88,88,...,131,131,131,131,131,131,131,131,132,131
440,5,99,77,77,77,78,79,79,79,79,...,86,85,85,85,85,85,85,85,85,85
441,5,100,114,114,116,120,124,125,125,125,...,195,195,195,195,195,196,197,199,200,200


In [12]:


# Read data: DUKE
#df2 = pd.read_csv(f'{wd_duke}{dk_dbh_fname}')
#df2_stack = pd.DataFrame(df2.iloc[:, :6].join(df2.iloc[:, 7:23].stack().reset_index(level=1, drop=True).rename('DBH')))
#df2_stack['ptime'] = pd.to_datetime(df2_stack['ind'].astype(str) + '1231', format="%DBH%Y%m%d")
#df2_stack.rename(columns={'treeid': 'treeid', 'RING': 'RING', 'co2': 'co2'}, inplace=True)
#df2_stack['BA'] = np.pi * (df2_stack['DBH'] / 2) ** 2

# Trees that die
#dead = df2_stack['treeid'][(df2_stack['ptime'] == df2_stack['ptime'].iloc[-1]) & (df2_stack['BA'] == 0)]
#df2_stack['die'] = df2_stack['treeid'].isin(dead)



# Save processed ORNL data
df2_stack.to_csv('ORNL-FACE_BasalArea_Norby.csv', index=False, quoting=False)

# Trees that die
dead = df2_stack['treeid'][(df2_stack['ptime'] == df2_stack['ptime'].iloc[-1]) & (df2_stack['BA'] == 0)]
df2_stack['die'] = df2_stack['treeid'].isin(dead)

df2_or = df2_stack
print(df2_or.head())

# Function to tidy the basal area (BA)
def tidy_ba(df2_stack):
    df2_stack['year'] = pd.Series(df2_stack['ptime'].dt.year)
    df2_stack_year = df2_stack.groupby(['RING', 'year', 'treeid'], as_index=False).agg({'BA': 'max'})
    df2_stack_year['co2'] = np.where(df2_stack_year['RING'].isin([1, 2]), 'ele', 'amb')

    df2_stack_year_diff = df2_stack_year.groupby(['co2', 'RING', 'treeid'], as_index=False).agg({'BA': lambda x: x.iloc[-1] - x.iloc[0]})
    df2_stack_year_1997_diff = df2_stack_year[df2_stack_year['year'] == 1997].merge(df2_stack_year_diff, on=['RING', 'treeid'])

    df2_stack_year_1997_diff_nomort = df2_stack_year_1997_diff[df2_stack_year_1997_diff['cBAI'] > 0]

    return df2_stack, df2_stack_year, df2_stack_year_diff, df2_stack_year_1997_diff, df2_stack_year_1997_diff_nomort

# Summarizing the data
df2_stack_sum = df2_stack.groupby(['ptime', 'RING'], as_index=False).agg({'BA': 'sum'})

df2_stack_rank = df2_stack.groupby(['RING', 'ptime'])\
    .apply(lambda x: x.assign(rank=pd.qcut(x['BA'], 4, labels=False, duplicates='drop')))\
    .reset_index(drop=True)
df2_stack_rank['BAfrac'] = df2_stack_rank['BA'] / df2_stack_rank.groupby(['RING', 'ptime'])['BA'].transform('sum')

df2_stack_rank_sum = df2_stack_rank.groupby(['RING', 'ptime'], as_index=False).agg({'BAfrac': 'sum'})

df2_stack_year_rank = df2_stack_year.groupby(['RING', 'year'], as_index=False)\
    .apply(lambda x: x.assign(rank=pd.qcut(x['BA'], 4, labels=False, duplicates='drop')))\
    .reset_index(drop=True)
df2_stack_year_rank['BAfrac'] = df2_stack_year_rank['BA'] / df2_stack_year_rank.groupby(['RING', 'year'])['BA'].transform('sum')

df2_stack_year_rank_sum = df2_stack_year_rank.groupby(['RING', 'year'], as_index=False).agg({'BAfrac': 'sum'})

df2_stack_year_rank_quartile = df2_stack_year_rank.groupby(['RING', 'co2', 'year', 'rank'], as_index=False).agg({'BAfrac': 'sum'})

df2_stack_year_rank_quartile_diff = df2_stack_year_rank_quartile.groupby(['co2', 'RING', 'rank'], as_index=False)\
    .agg({'deltaBAfrac': lambda x: x.iloc[-1] - x.iloc[0]})

df2_stack_count = df2_stack.groupby('ind').size().reset_index(name='count')
print(df2_stack_count.head(100))

df2_stack_ring = df2_stack.groupby(['ind', 'RING'], as_index=False).agg({'BA': 'sum'})
df2_stack_ring['ptime'] = pd.to_datetime(df2_stack_ring['ind'].str.replace('X', ''), format="%Y.%m.%d")
print(df2_stack_ring.head(100))

   TREE.ID Species  Plot  Quad  CO2     N   DBH
0    11003    PITA     1     1  amb  cont  18.9
0    11003    PITA     1     1  amb  cont  19.5
0    11003    PITA     1     1  amb  cont  20.4
0    11003    PITA     1     1  amb  cont  21.2
0    11003    PITA     1     1  amb  cont  21.8


KeyError: 'ind'